# S&P 500 Data Slide: Summary Statistics
FIM 500 GJR-GARCH project, midpoint check-in.

This notebook reproduces Parin Sharda's pipeline from `S&P500_datacleaned_Parin Sharda.ipynb` exactly (same ticker, window, log returns, and 8σ cleaning) and then computes the numbers for the Data slide. Parin's notebook is not modified.

Kernel: the project `.venv` (Python 3.11).

In [1]:
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 40)

## 1. Reproduce Parin's download, log returns, and 8σ cleaning

In [2]:
ticker = "^GSPC"
start_date = "2006-01-01"
end_date = "2026-09-20"

# Same call as Parin's notebook; yfinance auto-adjusts 'Close' by default
data = yf.download(ticker, start_date, end_date)['Close']
data = pd.DataFrame(data)
data.head()

[*********************100%***********************]  1 of 1 completed

Ticker,^GSPC
Date,
2006-01-03,1268.800049
2006-01-04,1273.459961
2006-01-05,1273.479980
2006-01-06,1285.449951
2006-01-09,1290.150024


In [3]:
# log returns: today vs yesterday
log_ret = np.log(data / data.shift(1)).dropna()

# Outlier cleaning, identical to Parin's: z-score on full-sample mean/std, 8σ cutoff
mean_r, std_r = log_ret.mean(), log_ret.std()
z = (log_ret - mean_r) / std_r
z = z["^GSPC"]
outliers = log_ret[z.abs() > 8]
log_ret_clean = log_ret[z.abs() < 8]

r_raw = log_ret["^GSPC"]       # uncleaned returns
r = log_ret_clean["^GSPC"]     # cleaned returns (used for all slide stats)

print(f"Flagged outliers: {len(outliers)}")
print(f"Returns before cleaning: {len(r_raw):,}   after cleaning: {len(r):,}")
assert len(outliers) == 4 and len(r) == 5205, "Does not match Parin's run (4 outliers, 5,205 returns)"

Flagged outliers: 4
Returns before cleaning: 5,209   after cleaning: 5,205


## 2. Flagged outliers: data errors or real market moves?
A data error usually shows up as a price that jumps and then snaps back the next day. A real move shows up as a lasting change in the price level. The previous and next closes are included so we can tell which one each flagged day is.

In [4]:
close = data["^GSPC"]
lo = (mean_r.iloc[0] - 8 * std_r.iloc[0]) * 100
hi = (mean_r.iloc[0] + 8 * std_r.iloc[0]) * 100
print(f"Cutoff |z| > 8  =>  log return outside [{lo:.2f}%, {hi:.2f}%]\n")

outlier_tbl = pd.DataFrame({
    "Prev close": close.shift(1).loc[outliers.index],
    "Close": close.loc[outliers.index],
    "Next close": close.shift(-1).loc[outliers.index],
    "Log return (%)": outliers["^GSPC"] * 100,
    "Simple return (%)": np.expm1(outliers["^GSPC"]) * 100,
    "z-score": z.loc[outliers.index],
})
outlier_tbl.index = outlier_tbl.index.date
outlier_tbl.index.name = "Date"
print(outlier_tbl.round(2).to_string())

Cutoff |z| > 8  =>  log return outside [-9.71%, 9.78%]

            Prev close    Close  Next close  Log return (%)  Simple return (%)  z-score
Date                                                                                   
2008-10-13      899.22  1003.35      998.01           10.96              11.58     8.96
2008-10-28      848.92   940.51      930.09           10.25              10.79     8.38
2020-03-12     2741.38  2480.64     2711.02           -9.99              -9.51    -8.23
2020-03-16     2711.02  2386.13     2529.19          -12.77             -11.98   -10.51


**Read:** all four are real, well-documented market moves, not data errors. The price level holds the next day in every case:
- 2008-10-13: +11.6% rally after the G7 and European bank-rescue announcements
- 2008-10-28: +10.8% rally
- 2020-03-12: −9.5%, market-wide circuit breaker (COVID)
- 2020-03-16: −12.0%, circuit breaker after the Fed's emergency cut to zero (COVID)

So the 8σ filter removes the two largest days of the 2008 crisis and the two largest days of the March 2020 crash. Those are the same events the 8σ threshold was chosen to keep. The rest of each crash period is still in the sample. This is worth a team decision before we fit the GJR-GARCH.

## 3. Sample window and observation counts

In [5]:
print(f"First price date          : {data.index[0].date()}")
print(f"Last price date           : {data.index[-1].date()}")
print(f"First return date         : {r.index[0].date()}   (raw: {r_raw.index[0].date()})")
print(f"Last return date          : {r.index[-1].date()}   (raw: {r_raw.index[-1].date()})")
print(f"Price observations        : {len(data):,}")
print(f"Return observations, raw  : {len(r_raw):,}")
print(f"Return observations, clean: {len(r):,}   ({len(outliers)} removed)")

First price date          : 2006-01-03
Last price date           : 2026-09-18
First return date         : 2006-01-04   (raw: 2006-01-04)
Last return date          : 2026-09-18   (raw: 2026-09-18)
Price observations        : 5,210
Return observations, raw  : 5,209
Return observations, clean: 5,205   (4 removed)


## 4. Descriptive statistics: cleaned series

In [6]:
def describe_returns(x):
    """Daily stats for a log-return series given in decimals; outputs marked _pct are in %."""
    jb = stats.jarque_bera(x)
    return {
        "mean_pct": x.mean() * 100,
        "std_pct": x.std() * 100,
        "ann_vol_pct": x.std() * np.sqrt(252) * 100,
        "min_pct": x.min() * 100,
        "min_date": x.idxmin().date(),
        "max_pct": x.max() * 100,
        "max_date": x.idxmax().date(),
        "skew": x.skew(),
        "ex_kurt": x.kurt(),  # pandas kurt() already returns excess (Fisher) kurtosis
        "jb_stat": jb.statistic,
        "jb_p": jb.pvalue,
    }

s_clean = describe_returns(r)

print(f"Mean daily return  : {s_clean['mean_pct']:.4f}%")
print(f"Daily std          : {s_clean['std_pct']:.4f}%")
print(f"Annualized vol     : {s_clean['ann_vol_pct']:.2f}%   (daily std x sqrt(252))")
print(f"Min daily return   : {s_clean['min_pct']:.2f}% on {s_clean['min_date']}")
print(f"Max daily return   : {s_clean['max_pct']:.2f}% on {s_clean['max_date']}")
print(f"Skewness           : {s_clean['skew']:.4f}")
print(f"Excess kurtosis    : {s_clean['ex_kurt']:.4f}")
print(f"Jarque-Bera stat   : {s_clean['jb_stat']:,.2f}")
print(f"Jarque-Bera p-value: {s_clean['jb_p']:.3e}")

Mean daily return  : 0.0348%
Daily std          : 1.1798%
Annualized vol     : 18.73%   (daily std x sqrt(252))
Min daily return   : -9.47% on 2008-10-15
Max daily return   : 9.09% on 2025-04-09
Skewness           : -0.4384
Excess kurtosis    : 8.9958
Jarque-Bera stat   : 17,679.00
Jarque-Bera p-value: 0.000e+00


## 5. Stationarity: Augmented Dickey-Fuller (cleaned series)

In [7]:
with warnings.catch_warnings():
    # statsmodels 0.15 warns the tuple return will change in 0.16; older versions reject result_object=
    warnings.simplefilter("ignore", FutureWarning)
    adf_res = adfuller(r, autolag="AIC")
adf_stat, adf_p, adf_lags, adf_nobs, adf_crit = adf_res[:5]

print(f"ADF statistic : {adf_stat:.4f}")
print(f"p-value       : {adf_p:.3e}")
print(f"Lags used     : {adf_lags}   (AIC-selected; nobs in regression = {adf_nobs:,})")
print("Critical values: " + ", ".join(f"{k}: {v:.3f}" for k, v in adf_crit.items()))

ADF statistic : -15.2389
p-value       : 5.196e-28
Lags used     : 19   (AIC-selected; nobs in regression = 5,185)
Critical values: 1%: -3.432, 5%: -2.862, 10%: -2.567


## 6. Ljung-Box: serial correlation in returns and squared returns (cleaned series)
The squared-returns test is the ARCH-effects check. Significant autocorrelation in $r_t^2$ means volatility clustering, which is what a GARCH-family model is built to capture.

In [8]:
lb_ret = acorr_ljungbox(r, lags=[10, 20])
lb_sq = acorr_ljungbox(r**2, lags=[10, 20])

lb_tbl = pd.concat({"Returns": lb_ret, "Squared returns": lb_sq}, names=["Series", "Lag"])
lb_tbl.columns = ["Q stat", "p-value"]
with pd.option_context("display.float_format", "{:.4g}".format):
    print(lb_tbl.to_string())

                     Q stat   p-value
Series          Lag                  
Returns         10    85.36 4.441e-14
                20    120.2 2.663e-16
Squared returns 10     4728         0
                20     6944         0


## 7. Uncleaned series, for comparison

In [9]:
s_raw = describe_returns(r_raw)

compare = pd.DataFrame(
    {
        "Cleaned": [s_clean["mean_pct"], s_clean["std_pct"], s_clean["skew"], s_clean["ex_kurt"]],
        "Uncleaned": [s_raw["mean_pct"], s_raw["std_pct"], s_raw["skew"], s_raw["ex_kurt"]],
    },
    index=["Mean daily return (%)", "Daily std (%)", "Skewness", "Excess kurtosis"],
)
print(compare.round(4).to_string())

                       Cleaned  Uncleaned
Mean daily return (%)   0.0348     0.0345
Daily std (%)           1.1798     1.2184
Skewness               -0.4384    -0.4753
Excess kurtosis         8.9958    12.9508


## 8. Summary table for the slide

In [10]:
def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def fmt_lb(row):
    p = fmt_p(row["lb_pvalue"])
    return f"{row['lb_stat']:,.1f} (p {p if p.startswith('<') else '= ' + p})"

price_span = f"{data.index[0].date()} to {data.index[-1].date()}"
ret_span = f"{r.index[0].date()} to {r.index[-1].date()}"
na = "—"

rows = [
    ("Price dates", price_span, price_span),
    ("Return dates", ret_span, f"{r_raw.index[0].date()} to {r_raw.index[-1].date()}"),
    ("Observations (daily log returns)", f"{len(r):,}", f"{len(r_raw):,}"),
    ("Outliers removed (|z| > 8)", f"{len(outliers)}", na),
    ("Mean daily return (%)", f"{s_clean['mean_pct']:.4f}", f"{s_raw['mean_pct']:.4f}"),
    ("Daily std (%)", f"{s_clean['std_pct']:.3f}", f"{s_raw['std_pct']:.3f}"),
    ("Annualized vol (%)", f"{s_clean['ann_vol_pct']:.2f}", f"{s_raw['ann_vol_pct']:.2f}"),
    ("Min daily return (%)", f"{s_clean['min_pct']:.2f} ({s_clean['min_date']})", f"{s_raw['min_pct']:.2f} ({s_raw['min_date']})"),
    ("Max daily return (%)", f"{s_clean['max_pct']:.2f} ({s_clean['max_date']})", f"{s_raw['max_pct']:.2f} ({s_raw['max_date']})"),
    ("Skewness", f"{s_clean['skew']:.3f}", f"{s_raw['skew']:.3f}"),
    ("Excess kurtosis", f"{s_clean['ex_kurt']:.2f}", f"{s_raw['ex_kurt']:.2f}"),
    ("Jarque-Bera stat", f"{s_clean['jb_stat']:,.0f}", na),
    ("Jarque-Bera p-value", fmt_p(s_clean['jb_p']), na),
    ("ADF stat (AIC lag selection)", f"{adf_stat:.2f}", na),
    ("ADF p-value", fmt_p(adf_p), na),
    ("ADF lags used", f"{adf_lags}", na),
    ("Ljung-Box Q(10), returns", fmt_lb(lb_ret.loc[10]), na),
    ("Ljung-Box Q(20), returns", fmt_lb(lb_ret.loc[20]), na),
    ("Ljung-Box Q(10), squared returns", fmt_lb(lb_sq.loc[10]), na),
    ("Ljung-Box Q(20), squared returns", fmt_lb(lb_sq.loc[20]), na),
]

summary = pd.DataFrame(rows, columns=["Statistic", "Cleaned (8σ)", "Uncleaned"]).set_index("Statistic")
summary

,Cleaned (8σ),Uncleaned
Statistic,,
Price dates,2006-01-03 to 2026-09-18,2006-01-03 to 2026-09-18
Return dates,2006-01-04 to 2026-09-18,2006-01-04 to 2026-09-18
Observations (daily log returns),"5,205","5,209"
Outliers removed (|z| > 8),4,—
Mean daily return (%),0.0348,0.0345
Daily std (%),1.180,1.218
Annualized vol (%),18.73,19.34
Min daily return (%),-9.47 (2008-10-15),-12.77 (2020-03-16)
Max daily return (%),9.09 (2025-04-09),10.96 (2008-10-13)
